# Tutorial 4: Fairness & Explainability Analysis

This tutorial demonstrates OneEHR's built-in fairness analysis and
feature importance methods.

In [ ]:
import numpy as np
import pandas as pd

## Fairness Analysis

OneEHR computes four fairness metrics across sensitive attributes:
- **Demographic parity**: equal positive prediction rate across groups
- **Equalized odds**: equal TPR and FPR across groups
- **Predictive parity**: equal PPV across groups
- **Standardized Mean Difference (SMD)**: effect size for prediction disparities

In [ ]:
from oneehr.analysis.fairness import compute_fairness

# Create synthetic predictions with demographic attributes
rng = np.random.default_rng(42)
n = 400

preds = pd.DataFrame({
    "patient_id": [f"p{i}" for i in range(n)],
    "system": ["xgboost"] * (n // 2) + ["gru"] * (n // 2),
    "y_true": rng.integers(0, 2, size=n).astype(float),
    "y_pred": rng.random(size=n),
})

static = pd.DataFrame({
    "patient_id": [f"p{i}" for i in range(n)],
    "age": rng.integers(20, 90, size=n),
    "sex": rng.choice(["M", "F"], size=n),
    "ethnicity": rng.choice(["White", "Black", "Asian", "Hispanic"], size=n),
})

result = compute_fairness(preds=preds, static=static)

print(f"Sensitive columns detected: {result['sensitive_columns']}")
for sys in result["systems"]:
    print(f"\nSystem: {sys['system']}")
    for attr, data in sys["attributes"].items():
        print(f"  {attr}:")
        print(f"    Demographic parity diff: {data['demographic_parity_diff']:.4f}")
        print(f"    Equalized odds diff: {data['equalized_odds_diff']:.4f}")
        if 'predictive_parity_diff' in data:
            print(f"    Predictive parity diff: {data['predictive_parity_diff']:.4f}")

## Feature Importance

OneEHR provides multiple feature importance methods:
1. **XGBoost native** (gain, cover, frequency)
2. **Permutation importance** (model-agnostic)
3. **SHAP** (Shapley Additive exPlanations)
4. **Integrated Gradients** (DL-specific)
5. **Attention weights** (for RETAIN, ConCare, Transformer)
6. **LIME** (Local Interpretable Model-agnostic Explanations)

In [ ]:
from oneehr.analysis.feature_importance import permutation_importance
from sklearn.ensemble import RandomForestClassifier

# Train a simple model
X = rng.random((200, 10))
y = (X[:, 0] + X[:, 1] > 1).astype(int)
feature_names = [f"feature_{i}" for i in range(10)]

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X, y)

result = permutation_importance(
    model, X, y,
    feature_names=feature_names,
    n_repeats=10,
)

print("Permutation Importance:")
for name, imp in sorted(zip(result.feature_names, result.importances), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.4f}")

In [ ]:
# Visualize feature importance
from oneehr.visualization.attribution import plot_waterfall

fig = plot_waterfall(
    result.importances,
    result.feature_names,
    top_k=10,
    title="Feature Importance (Permutation)",
    style="nature",
)
fig

## Attention Visualization

For attention-based models like RETAIN, you can extract and visualize
temporal attention weights:

In [ ]:
from oneehr.analysis.feature_importance import attention_importance

# Simulate attention weights (B, T) and input features (B, T, D)
attn_weights = rng.dirichlet([1] * 5, size=10)  # 10 patients, 5 time steps
X_seq = rng.random((10, 5, 8))  # 10 patients, 5 timesteps, 8 features

result = attention_importance(
    attn_weights, X_seq,
    feature_names=[f"feat_{i}" for i in range(8)],
)

print("Attention-weighted Feature Importance:")
for name, imp in sorted(zip(result.feature_names, result.importances), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.4f}")